In [1]:
# conda env 'vis2'

import sys
sys.path.append('/data/keeling/a/johntl3/b/LundstromPub2025/src')

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
import glob
import at3d

In [2]:
version_dir = '/data/keeling/a/johntl3/b/LundstromPub2025/CAMP2Ex_RF17/AT3D_output/V00'
def_files = np.sort(glob.glob(version_dir + '/default/iterations/iter*.nc'))
unc_files = np.sort(glob.glob(version_dir + '/CVM/darksurf_noCVM/iterations/iter*.nc'))
# unc_files = np.sort(glob.glob(version_dir + '/CVM/noCVM/iterations/iter*.nc'))
CVM_file = '/data/keeling/a/johntl3/b/LundstromPub2025/CAMP2Ex_RF17/intermediate_data/CVM_default.nc'

In [3]:
sensors_def, solvers_def, rte_grid_def = at3d.util.load_forward_model(def_files[-1])
sensors_unc, solvers_unc, rte_grid_unc = at3d.util.load_forward_model(unc_files[-1])

cloud_def = solvers_def[0.67].medium['cloud']
cloud_unc = solvers_unc[0.67].medium['cloud']

cvm = xr.load_dataset(CVM_file)
cvm['lcl_mask'].data = cvm['lcl_mask'].data.astype(np.int64)
cvm['cloud_volume_mask'].data = cvm['cloud_volume_mask'].data.astype(np.int64)
cvm['cth_mask'].data[0,:,:] = cvm['cth_mask'].data[-1,:,:] = 0
cvm['cth_mask'].data[:,0,:] = cvm['cth_mask'].data[:,-1,:] = 0

/data/keeling/a/johntl3/miniconda3/envs/vis2/lib/python3.10/site-packages/xarray/core/concat.py:500: FutureWarning: unique with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  common_dims = tuple(pd.unique([d for v in vars for d in v.dims]))
/data/keeling/a/johntl3/miniconda3/envs/vis2/lib/python3.10/site-packages/at3d/solver.py:2337: UserWarning: Number of SHDOM grid cells with optical depth greater than 2: '61'. Max cell optical depth: '3.3198599815368652'
  warnings.warn("Number of SHDOM grid cells with optical depth greater than 2: '{}'. "
/data/keeling/a/johntl3/miniconda3/envs/vis2/lib/python3.10/site-packages/xarray/core/concat.py:500: FutureWarning: unique with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  common_dims = tuple(pd.unique([d for v in vars for d in v.dims]))


In [4]:
cvm['cth_cloud'] = cvm.cth_mask & np.logical_not(cvm.lcl_mask) & np.logical_not(cvm.carved_mask)
cvm['lcl_cloud'] = np.logical_not(cvm.cth_mask) & cvm.lcl_mask & np.logical_not(cvm.carved_mask)
cvm['spc_cloud'] = np.logical_not(cvm.cth_mask) & np.logical_not(cvm.lcl_mask) & cvm.carved_mask
cvm['cth_clear'] = np.logical_not(cvm.cth_mask) & cvm.lcl_mask & cvm.carved_mask
cvm['lcl_clear'] = cvm.cth_mask & np.logical_not(cvm.lcl_mask) & cvm.carved_mask
cvm['spc_clear'] = cvm.cth_mask & cvm.lcl_mask & np.logical_not(cvm.carved_mask)
cvm['cloud'] = cvm.cth_mask & cvm.lcl_mask & cvm.carved_mask
cvm['clear'] = np.logical_not(cvm.cth_mask) & np.logical_not(cvm.lcl_mask) & np.logical_not(cvm.carved_mask)
cvm['clear'].data = cvm['clear'].data.astype(np.int64)
cvm['intersection'] = cvm.cth_cloud & cvm.lcl_cloud & cvm.spc_cloud & cvm.cth_clear & cvm.lcl_clear & cvm.spc_clear & cvm.cloud & cvm.clear
cvm['union'] = cvm.cth_cloud | cvm.lcl_cloud | cvm.spc_cloud | cvm.cth_clear | cvm.lcl_clear | cvm.spc_clear | cvm.cloud | cvm.clear


In [5]:
def segment_cloud(cloud, cvm):
    cloud['volume'] = cloud.extinction > 0
    cloud['volume'].data = cloud['volume'].data.astype(np.int64)
    cloud['cth_cloud'] = cloud.volume & cvm.cth_cloud
    cloud['lcl_cloud'] = cloud.volume & cvm.lcl_cloud
    cloud['spc_cloud'] = cloud.volume & cvm.spc_cloud
    cloud['cth_clear'] = cloud.volume & cvm.cth_clear
    cloud['lcl_clear'] = cloud.volume & cvm.lcl_clear
    cloud['spc_clear'] = cloud.volume & cvm.spc_clear
    cloud['cloud'] = cloud.volume & cvm.cloud
    cloud['clear'] = cloud.volume & cvm.clear
    cloud['intersection'] = cloud.cth_cloud & cloud.lcl_cloud & cloud.spc_cloud & cloud.cth_clear & cloud.lcl_clear & cloud.spc_clear & cloud.cloud & cloud.clear
    cloud['union'] = cloud.cth_cloud | cloud.lcl_cloud | cloud.spc_cloud | cloud.cth_clear | cloud.lcl_clear | cloud.spc_clear | cloud.cloud | cloud.clear
    return cloud


In [6]:
cloud_def = segment_cloud(cloud_def, cvm)
cloud_unc = segment_cloud(cloud_unc, cvm)

In [7]:
regions = ['cth_cloud', 'lcl_cloud', 'spc_cloud', 'cth_clear', 'lcl_clear', 'spc_clear', 'cloud', 'clear']
print('{:<10} | {:<38}    | {:<20}'.format('', 'Cloud Volume By Region (pct of total)', ''))
print('{:<10} | {:<19} | {:<19} | {:<20}'.format('Region', 'Constrained', 'Unconstrainted', 'CVM Volume'))
print('-'*80)
def_vol = cloud_def.volume.integrate(['x','y','z']).data
unc_vol = cloud_unc.volume.integrate(['x','y','z']).data
for r in regions:
    cvm_rvol = cvm[r].integrate(['x','y','z']).data
    def_rvol = cloud_def[r].integrate(['x','y','z']).data
    unc_rvol = cloud_unc[r].integrate(['x','y','z']).data
    def_rvol_pct = 100 * def_rvol / def_vol
    unc_rvol_pct = 100 * unc_rvol / unc_vol
    print('{:<10} | {:<8.3f} ({:<7.2f}%) | {:<8.3f} ({:<7.2f}%) | {:<20.3f}'.format(r, def_rvol, def_rvol_pct, unc_rvol, unc_rvol_pct, cvm_rvol))

           | Cloud Volume By Region (pct of total)     |                     
Region     | Constrained         | Unconstrainted      | CVM Volume          
--------------------------------------------------------------------------------
cth_cloud  | 0.000    (0.00   %) | 0.187    (0.12   %) | 14.068              
lcl_cloud  | 0.000    (0.00   %) | 1.845    (1.20   %) | 23.562              
spc_cloud  | 0.000    (0.00   %) | 0.000    (0.00   %) | 3.265               
cth_clear  | 0.000    (0.00   %) | 87.692   (57.18  %) | 201.732             
lcl_clear  | 0.000    (0.00   %) | 32.077   (20.92  %) | 104.290             
spc_clear  | 0.000    (0.00   %) | 0.251    (0.16   %) | 9.836               
cloud      | 34.685   (100.00 %) | 31.296   (20.41  %) | 76.179              
clear      | 0.000    (0.00   %) | 0.000    (0.00   %) | 0.732               


In [8]:
## Relative Bias and Difference from within the CVM
cond = cvm.cloud_volume_mask.data == 1
ext_def = cloud_def.extinction.data[cond]
ext_unc = cloud_unc.extinction.data[cond]
rel_diff = 100 * (np.sum((ext_unc - ext_def)**2))**0.5 / (np.sum(ext_def)**2)**0.5
rel_bias = 100 * (np.sum(np.abs(ext_unc)) - np.sum(np.abs(ext_def))) / (np.sum(np.abs(ext_def)))

print('Relative Bias: {:.2f}'.format(rel_bias))
print('Relative Diff: {:.2f}'.format(rel_diff))

Relative Bias: -42.30
Relative Diff: 1.50
